# 🗓️ Timeline Generator — MIMP
**Solo ejecuta todo (`Ctrl+F9`) y sube tu archivo cuando te lo pida.**

---

In [ ]:
# CELDA 1 — Instalación automática (no tocar)
import subprocess, sys, shutil, importlib
from pathlib import Path

# 1. Instalar dependencias
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "openpyxl", "python-pptx", "python-docx", "pdfplumber",
    "dateparser", "google-genai", "Pillow", "lxml"], check=True)

# 2. Descargar el código desde GitHub
REPO = "https://github.com/TU_USUARIO/timeline-generator-mimp"
ZIP_URL = f"{REPO}/raw/main/timeline_generator_v2.zip"

proj_dir = Path("/content/timeline_generator")
if proj_dir.exists():
    shutil.rmtree(proj_dir)

to_remove = [k for k in sys.modules if k.startswith(("core","utils","gui"))]
for k in to_remove: del sys.modules[k]

import urllib.request, zipfile
zip_path = Path("/content/timeline_generator_v2.zip")
print("⬇ Descargando código desde GitHub...")
urllib.request.urlretrieve(ZIP_URL, zip_path)

with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")

sys.path.insert(0, "/content/timeline_generator")
print("✓ Listo. No necesitas subir ningún ZIP.")


In [ ]:
# CELDA 2 — Cargar módulos
import importlib, logging
logging.basicConfig(level=logging.WARNING)

import core.model_analyzer;  importlib.reload(core.model_analyzer)
import core.file_parser;      importlib.reload(core.file_parser)
import core.timeline_builder; importlib.reload(core.timeline_builder)
import core.pptx_generator;   importlib.reload(core.pptx_generator)
import utils.date_detector;   importlib.reload(utils.date_detector)

from core.model_analyzer   import ModelAnalyzer
from core.file_parser      import FileParser
from core.timeline_builder import TimelineBuilder, BuilderConfig
from core.pptx_generator   import PptxGenerator
from utils.date_detector   import DetectorConfig
from datetime import date

template = ModelAnalyzer(
    "/content/timeline_generator/assets/formato_excel_modelo.xlsx"
).extract()

print(f"✓ Sistema listo | Fecha hoy: {date.today()}")
print("  Hito pasado → gris | En curso → azul | Futuro → amarillo")


In [ ]:
# CELDA 3 — Sube tu archivo
# Formatos: imagen (.png .jpg), Word (.docx), PDF (.pdf), Excel (.xlsx)
from google.colab import files as cf
from pathlib import Path

UPLOAD_DIR = Path("/content/mis_archivos")
UPLOAD_DIR.mkdir(exist_ok=True)

print("📂 Selecciona tu archivo de cronograma")
print("   (captura del SEACE, Word, PDF o Excel)")
uploaded = cf.upload()

SUPPORTED = {".xlsx",".xls",".docx",".pdf",".png",".jpg",".jpeg"}
INPUT_PATHS = []
for name, data in uploaded.items():
    ext = Path(name).suffix.lower()
    if ext not in SUPPORTED:
        print(f"  ⚠ Formato no soportado: {name}")
        continue
    dest = UPLOAD_DIR / name
    dest.write_bytes(data)
    INPUT_PATHS.append(dest)
    print(f"  ✓ {name}  ({len(data)//1024} KB)")

print(f"\n{len(INPUT_PATHS)} archivo(s) listo(s)")


In [ ]:
# CELDA 4 — Generar y descargar PowerPoint
from pathlib import Path
from google.colab import files as cf

USE_VISION = True   # usa Gemini Vision para leer imágenes

detect_cfg = DetectorConfig(granularity="annual")
build_cfg  = BuilderConfig(granularity="annual", max_columns=14)
parser     = FileParser(detect_cfg, use_vision=USE_VISION)
builder    = TimelineBuilder(build_cfg)

LAYOUTS = []
for filepath in INPUT_PATHS:
    print(f"\n📄 Procesando: {filepath.name}")
    try:
        parsed = parser.parse(filepath)
        layout = builder.build(parsed)
        LAYOUTS.append(layout)
        print(f"   Proyecto: {layout.project_title}")
        for s in layout.sections:
            print(f"   [{s.title}]")
            for e in s.columns[0].events if s.columns else []:
                d = e.detected_date.date_start.strftime("%d/%m/%Y") \
                    if e.detected_date and e.detected_date.date_start else "?"
                print(f"     [{e.status:7}] {d} | {e.label[:45]}")
    except Exception as ex:
        import traceback; traceback.print_exc()

if not LAYOUTS:
    print("❌ No se procesó ningún archivo.")
else:
    OUT = Path("/content/output"); OUT.mkdir(exist_ok=True)
    gen = PptxGenerator(template)
    for layout in LAYOUTS:
        gen.add_layout(layout)
    output_file = OUT / "timeline.pptx"
    gen.save(output_file)
    print(f"\n✅ PowerPoint generado")
    print("📥 Descargando...")
    cf.download(str(output_file))
    print("\n¡Listo! Revisa tu carpeta de Descargas.")
